# DS 208 &middot; Programming for Data Science &mdash; Week 7 Lab
## pandas II: Grouping, Merging &amp; Reshaping

This week the moves that do the real work: **group** to summarise by category,
**merge** to join two tables on a shared key, and **reshape** between long and wide.

**How long:** about 45 minutes. Nothing to install.

Work top to bottom. The Stretch section at the end is optional.

---
## Part 0 &middot; Group: split, apply, combine

Name the column to group on and the summary to apply; pandas returns one row per
group. Run the cell.

In [ ]:
import pandas as pd
df = pd.DataFrame({
    "region": ["Visayas", "Luzon", "Visayas", "Luzon", "Visayas"],
    "city":   ["Cebu", "Baguio", "Iloilo", "Manila", "Tacloban"],
    "pop":    [964_000, 366_000, 457_000, 1_780_000, 251_000],
})
print(df.groupby("region")["pop"].mean())

**Answer here** (double-click to edit):

1. Five rows went in; how many came out, and what decides that number?
   &rarr; *your answer*

2. The result is indexed by region, not by `0,1,2`. Why is that a convenient shape for a
   per-group summary?
   &rarr; *your answer*

---
## Part 1 &middot; Several summaries at once

Pass a list of summaries to `.agg()` and get a column for each &mdash; a compact
per-group report. Run the cell.

In [ ]:
report = df.groupby("region")["pop"].agg(["mean", "max", "count"])
print(report)

**Answer here:**

1. Read the `count` column. What does it count, and how does it relate to the number of
   input rows per region?
   &rarr; *your answer*

2. Add `"min"` to the list and rerun. What does the new column tell you?
   &rarr; *your answer*

---
## Part 2 &middot; Merge two tables on a key

Two tables share a `city` column &mdash; the **key**. Merging lines them up so you can
compute across both. Run the cell.

In [ ]:
areas = pd.DataFrame({
    "city":     ["Cebu", "Iloilo", "Manila"],
    "area_km2": [315, 78, 43],
})
both = pd.merge(df, areas, on="city")   # inner join by default
both["density"] = both["pop"] / both["area_km2"]
print(both[["city", "pop", "area_km2", "density"]])

**Answer here:**

1. The original `df` has five cities but `both` has fewer rows. Which cities dropped out,
   and why did an inner join remove them?
   &rarr; *your answer*

2. Change the merge to `how="left"` and rerun. What appears in `area_km2` for the cities
   that had no match, and what does that value mean?
   &rarr; *your answer*

---
## Part 3 &middot; Reshape long to wide

`pivot_table` spreads a category across columns; `melt` folds it back. Run the cell.

In [ ]:
sales = pd.DataFrame({
    "region": ["Visayas", "Visayas", "Luzon", "Luzon"],
    "year":   [2024, 2025, 2024, 2025],
    "sales":  [10, 14, 22, 25],
})
wide = sales.pivot_table(index="region", columns="year", values="sales", aggfunc="sum")
print(wide)

**Answer here:**

1. The long table had a `year` column; the wide one has years as column *headers*. Which
   layout is easier for a human to scan, and which is easier to group and plot?
   &rarr; *your answer*

2. Call `wide.reset_index().melt(id_vars="region", var_name="year", value_name="sales")`.
   What shape do you get back, and how does it compare to the original `sales`?
   &rarr; *your answer*

---
## Stretch &mdash; optional

Stop here if you like; the required part is done.

### Stretch 1 &middot; Group by two columns

Group `sales` by `["region", "year"]` and sum. What does the two-level index
represent?

In [ ]:
# your code here

### Stretch 2 &middot; Densest per region

Using `both` from Part 2, find the highest-density city in each region with one
`groupby(...)["density"].max()`.

In [ ]:
# your code here

---
## Submitting

Run the cell below. It uploads this notebook straight from Colab &mdash; nothing to
download.

You need a **submit token** &mdash; one covers every lab for a month. Open
[https://portal.latarak.com/student/submit-token](https://portal.latarak.com/student/submit-token), sign in and generate it,
then add it **once** to Colab's Secrets panel (the &#128273; icon, left sidebar) as
`LATARAK_TOKEN`. After that the cell reads it automatically, with no prompt. No Secrets
panel? The cell will just ask, hiding what you type.

In [ ]:
# --- Submit this notebook ------------------------------------------------------
# Colab only. Anywhere else, use the manual route described below this cell.
import getpass, json, urllib.request, urllib.error

PORTAL, COURSE, WEEK = "https://portal.latarak.com", "ds208", 7

try:
    from google.colab import _message
except ImportError:
    raise SystemExit(
        "Not running in Colab. Download this notebook "
        "(File > Download > Download .ipynb) and upload it at "
        "https://portal.latarak.com/course/ds208/lab/7/submit"
    )

# The LIVE notebook, including edits you have not saved yet.
nb = _message.blocking_request("get_ipynb", timeout_sec=90)["ipynb"]

# A month-long token. Store it once in Colab Secrets (key LATARAK_TOKEN) and
# this reads it with no prompt; otherwise it asks and hides what you type.
try:
    from google.colab import userdata
    token = (userdata.get("LATARAK_TOKEN") or "").strip()
except Exception:
    token = ""
if not token:
    token = getpass.getpass("Submit token (hidden as you type): ").strip()

req = urllib.request.Request(
    PORTAL + "/api/labs/" + COURSE + "/submit-notebook",
    data=json.dumps({"week": WEEK, "notebook": nb}).encode(),
    headers={"Content-Type": "application/json", "Authorization": "Bearer " + token},
    method="POST",
)
try:
    with urllib.request.urlopen(req, timeout=120) as r:
        out = json.load(r)
    print("Submitted", out["course"], "week", out["week"], "for", out["student"])
    print(out["cells"], "cells,", out["executed"], "executed")
    print(out["message"])
except urllib.error.HTTPError as e:
    print("Not submitted:", json.loads(e.read()).get("error", e.reason))

Prefer to do it by hand? **File &rarr; Download &rarr; Download .ipynb**, then go to the
[Week 7 submission page](https://portal.latarak.com/course/ds208/lab/7/submit) and upload it.

Re-submitting replaces your previous attempt; the most recent version is the one kept.